# Preprocessing Diabetes

Modul utama: `src/preprocess.py`. Notebook ini menjalankan modul yang sama, tanpa menduplikasi aturan cleaning. Target awal `<30` vs lainnya; keputusan lengkap di `docs/preprocessing.md`. Tidak melatih model.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src/preprocess.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/preprocess.py").exists(), "Buka dari root repo atau notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Jalankan ulang bila diperlukan

Data asli sudah disimpan lokal di data/raw. Ubah INPUT ke ZIP jika belum diekstrak. Set RUN_PIPELINE=True untuk menjalankan; default False membaca hasil yang sudah ada. Pilih kernel dari environment proyek.


In [ ]:
from src.preprocess import run

INPUT = ROOT / "data/raw/diabetic_data.csv"
RUN_PIPELINE = False
if RUN_PIPELINE:
    run(INPUT, ROOT / "data/processed/diabetes", ROOT / "results/preprocessing", "binary", 42)


## Ringkasan hasil agregat

Periksa jumlah baris, distribusi kelas, kolom dibuang dan overlap pasien. ID/data pasien tidak ditampilkan pada output notebook.


In [ ]:
report_path = ROOT / "results/preprocessing/preprocessing_summary.json"
assert report_path.exists(), "Jalankan pipeline terlebih dahulu"
summary = json.loads(report_path.read_text(encoding="utf-8"))
print("Raw:", summary["quality"]["raw_rows"], "Clean:", summary["quality"]["clean_rows"])
print("Target:", summary["target_mapping"])
print("Overlap pasien:", summary["patient_overlap_between_splits"])
print("Dropped:", summary["features"]["dropped_over_50_percent_missing_train"], summary["features"]["dropped_constant_train"])
pd.DataFrame(summary["splits"]).T


In [ ]:
pd.read_csv(ROOT / "results/preprocessing/missing_values.csv")


## Serah terima

Pemodelan memakai X_train.npz dan y_train.csv untuk model, validation untuk pemilihan parameter. Evaluasi memakai split yang sama untuk perbandingan. Test disimpan untuk penilaian final. Gunakan `ids_train.csv` sebagai groups untuk CV, jangan sebagai fitur; refit transformer di setiap fold. Lihat panduan untuk opsi multiclass dan keterbatasan cohort.
